# Optimisation multi-objectif (NSGA-II) — Équilibre thermique d'habitation

Reproduction **simplifiée** du notebook d'optimisation multi-objectif mentionné dans
[`BuildingTherm.mo`](BuildingTherm.mo) (sections citées en commentaire : météo, décalage
climatique...). Ce notebook :

- propose **le même choix de solveur** que l'app Streamlit — le modèle pur Python
  [`BuildingTherm.py`](BuildingTherm.py) (rapide, portable, aucune installation) ou le
  binaire **OpenModelica** compilé depuis [`BuildingTherm.mo`](BuildingTherm.mo) (solveur DAE
  de référence, installation plus longue) — voir section 0bis ;
- optimise les **mêmes 5 variables de conception** que les curseurs mis en avant dans l'app
  (Chauffage, Climatisation, Photovolt., Isolation ext., Isolation int.) ;
- calcule les **mêmes 3 indicateurs de sortie** que l'app (degrés-heures de manque de
  chauffage, degrés-heures d'excès de chaleur, coût €) ;
- lance **NSGA-II** ([`pymoo`](https://pymoo.org/)) pour explorer le compromis à 3 objectifs
  entre confort d'hiver, confort d'été et coût, avec des degrés-heures d'inconfort comme
  objectifs (plus lisses pour NSGA-II qu'un simple Tmin/Tmax : ils pèsent la durée du
  dépassement, pas seulement son pic) ;
- **regroupe** (k-means) le front de Pareto obtenu en **N scénarios représentatifs**
  (5 par défaut) et affiche, pour chacun, le même graphique temporel que l'app.

Voir le [dépôt GitHub](https://github.com/yannrichet/OptimHome) et le
[README](https://github.com/yannrichet/OptimHome#readme) pour le contexte complet
(modèle physique, app Streamlit, solveur de secours).


## 0. Installation (Google Colab) et récupération du modèle

Sur Colab, cette cellule installe les dépendances Python qui manquent et télécharge
`BuildingTherm.py` depuis GitHub (le notebook est autonome : il n'a pas besoin du
reste du dépôt cloné). En local, si `BuildingTherm.py` est déjà à côté de ce
notebook, l'installation est simplement ignorée.


In [1]:
import sys, subprocess, importlib.util, urllib.request

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "pymoo", "scikit-learn", "plotly", "pandas", "numpy", "scipy", "requests"],
        check=True,
    )

if importlib.util.find_spec("BuildingTherm") is None:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/yannrichet/OptimHome/main/BuildingTherm.py",
        "BuildingTherm.py",
    )
if importlib.util.find_spec("indicators") is None:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/yannrichet/OptimHome/main/indicators.py",
        "indicators.py",
    )

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from datetime import date, timedelta

import BuildingTherm as bt
from indicators import comfort_indicators

print("BuildingTherm.py + indicators.py chargés.")


BuildingTherm.py + indicators.py chargés.


## 0bis. Choix du solveur

Deux options, comme dans l'app Streamlit :

- `"python"` (par défaut) — [`BuildingTherm.py`](BuildingTherm.py), pur Python
  (`scipy.integrate.solve_ivp`, méthode `BDF`). Aucune installation, portable
  partout (Colab compris), ~4-5 s par simulation annuelle. Fidélité validée à
  ~0.02 % près par rapport à OpenModelica (voir le
  [README](https://github.com/yannrichet/OptimHome#solveur-de-secours-python-buildingthermpy)).
- `"openmodelica"` — compile et exécute le vrai modèle
  [`BuildingTherm.mo`](BuildingTherm.mo) via `omc`. Solveur DAE de référence
  (DASSL), généralement plus rapide *par simulation* une fois compilé, mais
  **installation d'OpenModelica sur Colab lente (~2-4 min)** et non garantie
  selon l'image Colab du moment.

Change simplement la valeur ci-dessous ; le reste du notebook s'adapte
automatiquement.


In [2]:
SOLVER = "python"   # "python" ou "openmodelica"


### Installation d'OpenModelica (uniquement si `SOLVER = "openmodelica"`)

Cellule sans effet si `SOLVER = "python"`. Sur Colab (Ubuntu), installe `omc`
depuis le dépôt APT officiel d'OpenModelica puis compile `BuildingTherm.mo`
(téléchargé depuis GitHub si absent) en un binaire, exactement comme
`build.mos` le fait dans le dépôt.


In [3]:
import shutil

if SOLVER == "openmodelica":
    if shutil.which("omc") is None:
        print("Installation d'OpenModelica (peut prendre 2-4 min)...")
        codename = subprocess.run(["lsb_release", "-cs"], capture_output=True, text=True).stdout.strip()
        subprocess.run(
            f"echo 'deb http://build.openmodelica.org/apt {codename} release' "
            "| sudo tee /etc/apt/sources.list.d/openmodelica.list",
            shell=True, check=True,
        )
        subprocess.run(
            "curl -fsSL http://build.openmodelica.org/apt/openmodelica.asc | sudo apt-key add -",
            shell=True, check=True,
        )
        subprocess.run("sudo apt-get update -qq", shell=True, check=True)
        subprocess.run("sudo apt-get install -y -qq --no-install-recommends omc", shell=True, check=True)
        print("OpenModelica installé.")
    else:
        print(f"omc déjà disponible : {shutil.which('omc')}")

    if not os.path.exists("BuildingTherm.mo"):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/yannrichet/OptimHome/main/BuildingTherm.mo",
            "BuildingTherm.mo",
        )
    with open("build_nb.mos", "w") as f:
        f.write('loadModel(Modelica); getErrorString();\n')
        f.write('loadFile("BuildingTherm.mo"); getErrorString();\n')
        f.write(
            'buildModel(BuildingTherm, outputFormat="csv", '
            'variableFilter="time|Eheat|Ecool|Egrid_cool|Eself_cool|Eexport|Tair|Tout|Qheat|Pelec|Ppv|Pself_cool|Pgrid_cool"); '
            'getErrorString();\n'
        )
    subprocess.run(["omc", "build_nb.mos"], check=True)
    print("Binaire BuildingTherm compilé.")
else:
    print("SOLVER='python' : aucune installation supplémentaire nécessaire.")


SOLVER='python' : aucune installation supplémentaire nécessaire.


## 1. Météo réelle (Open-Meteo)

Même source et même API que l'app (`archive-api.open-meteo.com`, réanalyse ERA5,
sans clé). Position et période par défaut ci-dessous — à changer librement.
Le fichier `weather_nb.txt` n'est écrit que pour le solveur OpenModelica
(`CombiTimeTable` lit un fichier ; le solveur Python prend les tableaux
directement en mémoire).


In [4]:
LAT, LON = 48.8566, 2.3522          # Paris par défaut ; change librement
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=364)
WARMUP_DAYS = 14                     # mise en régime, comme dans l'app


def fetch_weather(lat, lon, start_date, end_date, warmup_days=WARMUP_DAYS):
    """(times, Tout_K, Gh) horaires, même format que BuildingTherm.load_weather_table."""
    fetch_start = start_date - timedelta(days=warmup_days)
    r = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={"latitude": lat, "longitude": lon,
                "start_date": fetch_start.isoformat(), "end_date": end_date.isoformat(),
                "hourly": "temperature_2m,shortwave_radiation", "timezone": "UTC"},
        timeout=60,
    )
    r.raise_for_status()
    h = r.json()["hourly"]
    Tout_K = np.array(h["temperature_2m"]) + 273.15
    Gh = np.maximum(np.array(h["shortwave_radiation"]), 0.0)
    times = np.arange(len(Tout_K)) * 3600.0
    return times, Tout_K, Gh


def write_weather_file(weather, path):
    """Meme format texte que celui ecrit par app.py (table CombiTimeTable)."""
    times, Tout_K, Gh = weather
    with open(path, "w") as f:
        f.write("#1\n")
        f.write(f"double tmy({len(times)},3)\n")
        for t, tk, g in zip(times, Tout_K, Gh):
            f.write(f"{t:.0f} {tk:.2f} {max(g, 0.0):.1f}\n")


weather = fetch_weather(LAT, LON, START_DATE, END_DATE)
stop_time = (len(weather[0]) - 1) * 3600.0
print(f"{len(weather[0])} points horaires ({stop_time/86400:.1f} j, dont {WARMUP_DAYS} j de mise en régime)")

WEATHER_PATH = os.path.abspath("weather_nb.txt")
if SOLVER == "openmodelica":
    write_weather_file(weather, WEATHER_PATH)
    print(f"Météo écrite dans {WEATHER_PATH}")


9096 points horaires (379.0 j, dont 14 j de mise en régime)


## 2. Variables de conception, objectifs, fonction de simulation

**Variables de conception** (5, mêmes bornes que les sliders de l'app) :
`Pheat` (Chauffage), `Pcool` (Climatisation), `Ppv_kWc` (Photovolt.),
`e_ite_cm` (Isolation ext.), `e_iti_cm` (Isolation int.).

**Objectifs** (3, mêmes indicateurs que l'app, tous à **minimiser**) — des degrés-heures
d'inconfort plutôt qu'un simple Tmin/Tmax : plus lisses pour NSGA-II (Tmin/Tmax sont des
statistiques d'ordre à gradient quasi partout nul ; une somme varie avec chaque variable de
conception), et plus représentatifs du vécu réel (un pic bref pèse moins qu'un dépassement
prolongé) :
- **Froid·Heure** [K·h] : `Σ max(0, T_confort_min − Tair_h)` sur toutes les heures de la
  période (après mise en régime) — degrés-heures sous la limite basse (≈ DJU horaires).
- **Chaleur·Heure** [K·h] : `Σ max(0, Tair_h − T_confort_max)` — degrés-heures au-dessus de
  la limite haute.
- **Coût net** [€] : `Egrid_cool·prix_elec − Eexport·prix_rachat_pv`, sur la période choisie.

`simulate_scenario()` appelle le solveur Python ou le binaire OpenModelica selon
`SOLVER`, mais renvoie dans les deux cas un DataFrame avec les mêmes colonnes
(`time, Tair, Tout, Qheat, Pgrid_cool, Pself_cool, Egrid_cool, Eself_cool, Eexport`) —
tout le reste du notebook (objectifs, graphiques) est indépendant du solveur choisi.

Tous les autres paramètres du modèle restent aux valeurs par défaut de l'app
(matériau parpaing, géométrie 40 m²/7,5 m, ventilation, etc.).


In [5]:
FIXED_PARAMS = dict(
    ach_day=1.5, ach_night=2.0,
    lam_iso=0.036, ach=0.6, Qint=400.0, dTout=1.0, fsol=0.5, seer=3.5,
    Sfloor=40.0, Htot=7.5, Awin=20.0, UAother_ref=58.0, Sfloor_ref=40.0,
    e_blk=0.20, lam_blk=0.95, rhoc_blk=1300 * 1000.0, rhoc_iso=30 * 1030.0,
    hi=7.7, he=25.0,
    Tset_h=292.15, Tset_c=299.15, Kp=4000.0, Kc=4000.0,
    fanWhm3=0.15, PR_pv=0.90,
)
PRIX_ELEC, PRIX_RACHAT_PV = 0.2516, 0.04     # €/kWh, mêmes valeurs par défaut que l'app
T_CONFORT_MIN, T_CONFORT_MAX = 19.0, 26.0    # °C, bande de confort par defaut de l'app
WARMUP_HOURS = WARMUP_DAYS * 24

VAR_NAMES = ["Pheat", "Pcool", "Ppv_kWc", "e_ite_cm", "e_iti_cm"]
VAR_LABELS = ["Chauffage [W]", "Climatisation [W]", "Photovolt. [kWc]",
              "Isolation ext. [cm]", "Isolation int. [cm]"]
XL = np.array([1000.0, 0.0, 0.0, 0.0, 0.0])       # mêmes bornes que les sliders de l'app
XU = np.array([12000.0, 6000.0, 9.0, 30.0, 20.0])

OBJ_NAMES = ["DH_froid_Kh", "DH_chaleur_Kh", "Cout_net_eur"]   # tous a minimiser directement
OBJ_LABELS = ["Froid·Heure [K·h]", "Chaleur·Heure [K·h]", "Coût net [€]"]

OUTPUT_COLS = ["time", "Tair", "Tout", "Qheat", "Pgrid_cool", "Pself_cool",
               "Egrid_cool", "Eself_cool", "Eexport"]


def _scenario_params(x):
    Pheat, Pcool, Ppv_kWc, e_ite_cm, e_iti_cm = x
    return dict(FIXED_PARAMS, Pheat=Pheat, Pcool=Pcool, Ppv_kWc=Ppv_kWc,
                e_ite=e_ite_cm / 100.0, e_iti=e_iti_cm / 100.0)


def _simulate_python(x):
    rows = bt.simulate(_scenario_params(x), weather, stop_time)
    return pd.DataFrame(rows)[OUTPUT_COLS]


def _simulate_openmodelica(x):
    """Appelle le binaire compile, comme app_fzr/run.sh mais sans passer par fz/fzr
    (inutile ici : un seul cas a la fois, pas de contexte Streamlit multi-thread)."""
    override = f"tmy.fileName={WEATHER_PATH}," + ",".join(f"{k}={v}" for k, v in _scenario_params(x).items())
    subprocess.run(
        ["./BuildingTherm", f"-override={override}", "-startTime=0",
         f"-stopTime={stop_time}", "-stepSize=3600", "-r=res_nb.csv"],
        check=True, capture_output=True, text=True,
    )
    raw = pd.read_csv("res_nb.csv")
    # meme nettoyage que app.py : plusieurs lignes quasi simultanees autour des
    # evenements de ventilation -> on garde une ligne par heure (derniere valeur)
    raw["hour"] = (raw["time"] / 3600).round().astype(int)
    sim = raw.groupby("hour", as_index=False).last()
    sim["time"] = sim["hour"] * 3600
    return sim.drop(columns="hour")[OUTPUT_COLS]


def simulate_scenario(x):
    """x = [Pheat, Pcool, Ppv_kWc, e_ite_cm, e_iti_cm] -> DataFrame (colonnes = app)."""
    if SOLVER == "openmodelica":
        return _simulate_openmodelica(x)
    return _simulate_python(x)


def objectives(x):
    """(DH_froid, DH_chaleur, cout_net_eur) sur la periode, apres mise en regime — les 3 a minimiser.

    DH_froid/DH_chaleur = degres-heures d'inconfort (Sum des depassements horaires de la bande
    de confort, en K.h — meme fonction que l'app, indicators.comfort_indicators) plutot que
    Tmin/Tmax : une somme, contrairement a un min/max, varie avec chaque variable de conception
    (meilleur signal pour NSGA-II) et pese la duree du depassement, pas seulement son pic.
    """
    sim = simulate_scenario(x)
    sim_p = sim.iloc[WARMUP_HOURS:] if len(sim) > WARMUP_HOURS else sim
    Tint = sim_p["Tair"] - 273.15
    _, DH_froid, DH_chaleur = comfort_indicators(Tint, T_CONFORT_MIN, T_CONFORT_MAX)
    egrid_cool = sim["Egrid_cool"].iloc[-1]
    eexport = sim["Eexport"].iloc[-1]
    cout = egrid_cool * PRIX_ELEC - eexport * PRIX_RACHAT_PV
    return DH_froid, DH_chaleur, cout


# sanity check sur un scenario "par defaut" (memes valeurs que l'app au chargement)
print(f"solveur = {SOLVER!r}")
print("objectifs (defaut app) [DH_froid, DH_chaleur, Cout] :", objectives([8000, 2000, 3.0, 16.0, 0.0]))


solveur = 'python'


objectifs (defaut app) [DH_froid, DH_chaleur, Cout] : (np.float64(789.1365804964148), np.float64(2935.265346882262), np.float64(681.960345284624))


## 3. Optimisation multi-objectif (NSGA-II, `pymoo`)

`POP_SIZE` × `N_GEN` simulations annuelles complètes seront exécutées (chacune
~3 s en pur Python, généralement plus rapide avec le binaire OpenModelica
déjà compilé) : avec les valeurs par défaut ci-dessous (pop=40, 20 générations, soit ~800
simulations annuelles), compter environ 35-45 minutes en pur Python. Augmenter ces deux valeurs donne un front de Pareto plus fin, au prix
du temps de calcul (linéaire).


In [6]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize


class BuildingProblem(Problem):
    def __init__(self):
        super().__init__(n_var=5, n_obj=3, xl=XL, xu=XU)

    def _evaluate(self, X, out, *args, **kwargs):
        out["F"] = np.array([objectives(x) for x in X])


POP_SIZE = 40   # reduire (ex. 16) pour un run plus rapide, augmenter pour un front plus fin
N_GEN = 20        # reduire (ex. 6) pour un run plus rapide, augmenter pour une meilleure convergence

algorithm = NSGA2(
    pop_size=POP_SIZE,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(eta=20),
)

res = minimize(BuildingProblem(), algorithm, ("n_gen", N_GEN), seed=1, verbose=True)
print(f"{len(res.F)} solutions non dominées sur le front de Pareto final")


n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       40 |     13 |             - |             -


     2 |       80 |     23 |  0.0035115837 |         ideal


     3 |      120 |     22 |  0.1760354900 |         ideal


     4 |      160 |     20 |  0.0127708206 |         ideal


     5 |      200 |     24 |  0.0458769986 |         ideal


...

    16 |      640 |     40 |  0.0650569674 |         nadir


    17 |      680 |     40 |  0.0154114242 |         ideal


    18 |      720 |     40 |  0.0698740733 |         nadir


    19 |      760 |     40 |  0.0085514173 |         ideal


    20 |      800 |     40 |  0.0185142040 |         nadir
40 solutions non dominées sur le front de Pareto final


## 4. Regroupement du front de Pareto en N scénarios représentatifs

K-means (dans l'espace des 3 objectifs, standardisé) découpe le front en
`N_SCENARIOS` groupes ; pour chacun, on retient la solution la plus proche du
centroïde comme scénario représentatif.


In [7]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

N_SCENARIOS = 5   # nombre de scenarios a extraire du front de Pareto

F, X = res.F, res.X   # F : (n_sol, 3) [DH_froid_Kh, DH_chaleur_Kh, cout_net_eur] ; X : (n_sol, 5) variables

scaler = StandardScaler()
F_scaled = scaler.fit_transform(F)

k = min(N_SCENARIOS, len(F))
kmeans = KMeans(n_clusters=k, n_init=10, random_state=0).fit(F_scaled)

selected_idx = []
for c in range(k):
    members = np.where(kmeans.labels_ == c)[0]
    center = kmeans.cluster_centers_[c]
    dists = np.linalg.norm(F_scaled[members] - center, axis=1)
    selected_idx.append(members[np.argmin(dists)])
selected_idx = sorted(selected_idx, key=lambda i: F[i, 2])  # tri par cout croissant

scenarios_df = pd.DataFrame(X[selected_idx], columns=VAR_NAMES)
scenarios_df["DH_froid_Kh"] = F[selected_idx, 0]
scenarios_df["DH_chaleur_Kh"] = F[selected_idx, 1]
scenarios_df["Cout_net_eur"] = F[selected_idx, 2]
scenarios_df.index = [f"Scénario {i + 1}" for i in range(len(selected_idx))]
scenarios_df.round(1)


,Pheat,Pcool,Ppv_kWc,e_ite_cm,e_iti_cm,DH_froid_Kh,DH_chaleur_Kh,Cout_net_eur
Scénario 1,1116.6,651.4,9.0,29.8,0.0,2763.2,6909.5,107.7
Scénario 2,1149.4,5499.0,8.7,19.2,0.0,3334.5,838.7,191.0
Scénario 3,1693.0,5017.1,8.9,19.7,0.0,1297.7,849.8,259.7
Scénario 4,8540.9,5487.6,5.0,19.1,0.0,739.6,838.5,506.5
Scénario 5,1298.0,5293.1,5.1,11.2,0.0,3899.4,810.9,530.8


## 5. Front de Pareto (3D) : degrés-heures froid / chaleur / coût

In [8]:
DH_froid_all, DH_chaleur_all, Cout_all = F[:, 0], F[:, 1], F[:, 2]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=DH_froid_all, y=DH_chaleur_all, z=Cout_all, mode="markers", name="Front de Pareto",
    marker=dict(color="rgba(31,119,180,0.5)", size=4),
))
fig.add_trace(go.Scatter3d(
    x=DH_froid_all[selected_idx], y=DH_chaleur_all[selected_idx], z=Cout_all[selected_idx],
    mode="markers+text", name="Scénarios retenus",
    marker=dict(color="rgba(214,39,40,0.9)", size=7, symbol="diamond"),
    text=[f"S{i + 1}" for i in range(len(selected_idx))],
))
fig.update_layout(
    scene=dict(xaxis_title="Froid·Heure [K·h]", yaxis_title="Chaleur·Heure [K·h]", zaxis_title="Coût net [€]"),
    title="Front de Pareto à 3 objectifs — scénarios représentatifs en évidence",
    height=550, margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()


## 6. Détail temporel des scénarios sélectionnés

Même visualisation que l'app Streamlit (bande de confort, températures
intérieure/extérieure min-max journalières, puissances importées/autoconsommées).


In [9]:
def plot_scenario(x, title):
    sim = simulate_scenario(x)
    sim_p = sim.iloc[WARMUP_HOURS:].reset_index(drop=True)
    jours = ((sim_p["time"] - sim_p["time"].iloc[0]) / 86400).astype(int)

    daily = pd.DataFrame({
        "jour": jours,
        "Tint_min": sim_p["Tair"] - 273.15, "Tint_max": sim_p["Tair"] - 273.15,
        "Text_min": sim_p["Tout"] - 273.15, "Text_max": sim_p["Tout"] - 273.15,
    }).groupby("jour").agg({"Tint_min": "min", "Tint_max": "max", "Text_min": "min", "Text_max": "max"})
    kW_grid = (sim_p["Pgrid_cool"] / 1000).groupby(jours).mean()
    kW_pv_self = (sim_p["Pself_cool"] / 1000).groupby(jours).mean()

    fig = go.Figure()
    fig.add_hrect(y0=T_CONFORT_MIN, y1=T_CONFORT_MAX, fillcolor="rgba(46,160,67,0.12)", line_width=0,
                  annotation_text=f"confort {T_CONFORT_MIN:.0f}-{T_CONFORT_MAX:.0f} °C", annotation_position="top left")
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Text_max"], mode="lines",
                              line=dict(width=1.2, color="rgba(120,120,120,0.9)"), name="T extérieure max/j"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Text_min"], mode="lines",
                              line=dict(width=1.2, color="rgba(120,120,120,0.9)", dash="dot"),
                              fill="tonexty", fillcolor="rgba(120,120,120,0.25)", name="T extérieure min/j"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Tint_max"], mode="lines",
                              line=dict(width=1.2, color="rgba(31,119,180,0.9)"), name="T intérieure max/j"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Tint_min"], mode="lines",
                              line=dict(width=1.2, color="rgba(31,119,180,0.9)", dash="dot"),
                              fill="tonexty", fillcolor="rgba(31,119,180,0.3)", name="T intérieure min/j"))
    fig.add_trace(go.Scatter(x=kW_grid.index, y=kW_grid.values, mode="lines", name="Import réseau [kW]",
                              stackgroup="power", yaxis="y2", line=dict(color="rgba(214,39,40,0.9)", width=0.5),
                              fillcolor="rgba(214,39,40,0.35)"))
    fig.add_trace(go.Scatter(x=kW_pv_self.index, y=kW_pv_self.values, mode="lines", name="Autoconso PV [kW]",
                              stackgroup="power", yaxis="y2", line=dict(color="rgba(255,127,14,0.9)", width=0.5),
                              fillcolor="rgba(255,127,14,0.35)"))
    fig.update_layout(
        title=title, xaxis_title="Jour", yaxis=dict(title="Température [°C]"),
        yaxis2=dict(title="Puissance moyenne/j [kW]", overlaying="y", side="right", rangemode="tozero"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        height=380, margin=dict(l=60, r=60, t=60, b=40), hovermode="x unified",
    )
    return fig


for i, idx in enumerate(selected_idx):
    x = X[idx]
    params_txt = ", ".join(f"{n}={v:.1f}" for n, v in zip(VAR_NAMES, x))
    title = (f"Scénario {i + 1}/{len(selected_idx)} — {params_txt}<br>"
             f"Froid·Heure {F[idx, 0]:.0f} K·h · Chaleur·Heure {F[idx, 1]:.0f} K·h · Coût net {F[idx, 2]:.0f} €")
    plot_scenario(x, title).show()
